# CSPN samples and metrics

What the circuit draws, and how good it is at drawing it. One sampling pass per model over
all 180 `(digit, fg, bg)` combinations feeds every metric and every figure here, so a
number and a picture always describe the same samples.

**The metrics**

| metric | what it answers | reference |
|---|---|---|
| `colour/fg_accuracy`, `colour/bg_accuracy` | did it use the colours it was asked for? | 1.0 |
| `colour/*_drift` | how far off, once accuracy saturates | 0.0 |
| `colour/contrast` | is there a digit at all, or a flat wash? | ~0.89–0.94 on real data |
| `digit/accuracy` | does the judge read the requested digit? | **ceiling 0.971**, floor 0.0996 |
| `diversity/*` | mean pairwise distance among samples of one label | real latents ≈ 3.148 |
| `latent/mahalanobis` | are sampled latents where real ones live? | ~1.0× real |
| `nll/*` | density on real held-out latents | lower is better |
| `discrimination/*` | does the model score the true label above the others? | 1.0 |

**Two traps this table exists to avoid.** Colour fidelity alone rises monotonically as
sampling gets more mode-proximate, so it must always be read next to `diversity`. And a
digit accuracy of 0.5 is halfway to the achievable ceiling of 0.971, not "half wrong".

**The held-out column is empty on `uniform` and `skewed`.** Neither weight table zeroes a
cell, so every combination was trained on and `split()` returns NaN for the held-out half.
It only fills in on a variant that actually holds combinations out.

**NLL is not comparable across autoencoders.** Two CSPNs trained on different latent
spaces are being asked about different random variables; compare them on the sampled-image
metrics instead.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from dataset_loaders.colour_mnist import ColourMNIST, seen_mask
from evaluation import (
    ColourFidelity,
    DigitAccuracy,
    LabelDiscrimination,
    LatentPlausibility,
    NegativeLogLikelihood,
    SampleDiversity,
    load_digit_classifier,
    run_eval_suite,
    sample_combination_grid,
    sample_for_label,
)
from utils import resolve_device
from utils.checkpoints import (
    load_ae_from_path,
    load_cspn_from_path,
    load_joint_pc_from_path,
    load_nn_baseline_from_path,
)
from utils.visualisation import (
    plot_combination_grid,
    plot_combination_heatmap,
    plot_image_rows,
)
from utils.wandb_utils import load_from_wandb

LOADERS = {
    "cspn": load_cspn_from_path,
    "joint_pc": load_joint_pc_from_path,
    "nn_baseline": load_nn_baseline_from_path,
}


@dataclass
class Spec:
    label: str
    kind: str
    name: str
    ae: str
    tag: str = "latest"
    ae_tag: str = "latest"


# NOTE: a CSPN's artifact name is built from its dataset, not its autoencoder, so both
# runs below live in the *same* collection and are told apart only by version. Check the
# versions against wandb before trusting the labels.
MODELS = [
    Spec("CSPN · plain VAE", "cspn", "psinet_colour_mnist_uniform",
         "variational_colour_mnist_uniform", tag="v1"),
    Spec("CSPN · supervised VAE", "cspn", "psinet_colour_mnist_uniform",
         "supervised_colour_mnist_uniform", tag="latest"),
    # Spec("joint PC", "joint_pc", "joint_pc_colour_mnist_uniform",
    #      "variational_colour_mnist_uniform"),
]

VARIANT = "uniform"
SAMPLES_PER_COMBINATION = 64
STD_CORRECTION = 0.6  # matches real latent dispersion; 1.0 is over-dispersed
FOCUS_LABEL = (3, 0, 1)  # (digit 3, red, black) -- the strip everything is compared on
DENSITY_BATCHES = 8
BATCH_SIZE = 256
FIGURES = Path("artifacts/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

device = resolve_device()
device = torch.device("cpu")

In [ ]:
def loader(split: str) -> DataLoader:
    dataset = ColourMNIST(
        root="../data",
        split=split,
        variant=VARIANT,
        transform=transforms.Compose([transforms.ToTensor()]),
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


@torch.no_grad()
def reference_latents(ae) -> np.ndarray:
    return np.concatenate(
        [ae.encode(images.to(device)).cpu().numpy() for images, _ in loader("train")]
    )


def load(spec: Spec):
    model = LOADERS[spec.kind](
        load_from_wandb(spec.name, spec.tag), device=device
    ).to(device)
    ae = load_ae_from_path(
        load_from_wandb(spec.ae, spec.ae_tag), device=device
    ).to(device)
    return model, ae


judge = load_digit_classifier(device=device)
seen = seen_mask("../data", VARIANT)
test_loader = loader("test")
loaded = {spec.label: load(spec) for spec in MODELS}
list(loaded)

## Metrics

In [ ]:
reports = {}
for label, (model, ae) in loaded.items():
    reports[label] = run_eval_suite(
        model,
        ae,
        device,
        sample_metrics=[
            ColourFidelity(),
            DigitAccuracy(judge),
            SampleDiversity(SAMPLES_PER_COMBINATION),
            LatentPlausibility(reference_latents(ae)),
        ],
        density_metrics=[NegativeLogLikelihood(), LabelDiscrimination()],
        loader=test_loader,
        seen=seen,
        samples_per_combination=SAMPLES_PER_COMBINATION,
        std_correction=STD_CORRECTION,
        max_density_batches=DENSITY_BATCHES,
    )

summary = pd.DataFrame({label: report.summary() for label, report in reports.items()})
summary.round(4)

In [ ]:
# Trained vs held-out, for the variants that hold combinations out at all.
overall = pd.DataFrame(
    {
        label: {name: report.overall(name) for name in sorted(report.tables)}
        for label, report in reports.items()
    }
)
overall.round(4)

### Where a gap lives

The per-combination table says *which* combinations a model is bad at; the marginals say
whether a gap follows the digit axis, the foreground axis or the background axis.

In [ ]:
METRIC = "colour/bg_accuracy"

for label, report in reports.items():
    figure = plot_combination_heatmap(
        report.tables[METRIC],
        held_out=None if report.seen is None else ~report.seen,
        title=f"{METRIC} — {label}", vmin=0.0, vmax=1.0,
    )
    figure.savefig(FIGURES / f"heatmap_{METRIC.replace('/', '_')}_{label.replace(' ', '_')}.png",
                   bbox_inches="tight")

for label, report in reports.items():
    axes = report.marginals(METRIC)
    print(label, {axis: np.round(values, 3) for axis, values in axes.items()})

## Samples for every combination

Digits down the rows, colours across: within a background group the six columns are the
foreground palette. Same layout as the heatmap above, so a suspicious cell can be looked at
directly.

In [ ]:
grids = {}
for label, (model, ae) in loaded.items():
    grids[label] = sample_combination_grid(
        model, ae, device, samples_per_combination=1, std_correction=STD_CORRECTION
    )
    figure = plot_combination_grid(
        grids[label][:, :, :, 0], title=f"{label}"
    )
    figure.savefig(FIGURES / f"grid_{label.replace(' ', '_').replace('·', '-')}.png",
                   bbox_inches="tight")

## One combination, side by side

The grid above shows one draw per cell, which cannot show whether a model repeats itself.
This strip fixes a single label and asks each model for several samples, with real images
of that same combination on top as the reference.

In [ ]:
COUNT = 10

dataset = ColourMNIST(
    root="../data", split="test", variant=VARIANT,
    transform=transforms.Compose([transforms.ToTensor()]),
)
matching = (dataset.targets == torch.tensor(FOCUS_LABEL)).all(dim=1).nonzero().flatten()
real = torch.stack([dataset[int(i)][0] for i in matching[:COUNT]])

rows = {"real": real}
for label, (model, ae) in loaded.items():
    rows[label] = sample_for_label(
        model, ae, FOCUS_LABEL, COUNT, device, std_correction=STD_CORRECTION
    )

figure = plot_image_rows(rows, title=f"digit {FOCUS_LABEL[0]}, fg {FOCUS_LABEL[1]}, bg {FOCUS_LABEL[2]}")
figure.savefig(FIGURES / "focus_label.png", bbox_inches="tight")

### How much of that spread is the sampler?

`std_correction` scales the within-component width only. Sweeping it on one label shows
what the fidelity metrics are trading away: the colours get cleaner as it drops, which is
exactly why fidelity must never be tuned on its own.

In [ ]:
SWEEP = [1.0, 0.6, 0.3, 0.1]
label, (model, ae) = next(iter(loaded.items()))

rows = {
    f"std {std}": sample_for_label(model, ae, FOCUS_LABEL, COUNT, device, std_correction=std)
    for std in SWEEP
}
figure = plot_image_rows(rows, title=f"{label} — std_correction sweep on one label")
figure.savefig(FIGURES / "std_correction_sweep.png", bbox_inches="tight")